# Clustering des séries temporelles de consommation énergétique des bâtiments individuels

## Objectif du notebook

L'objectif de ce notebook est d'analyser les séries temporelles de consommation énergétique des bâtiments individuels afin d'identifier des **jours types de consommation**.

L'idée est de regrouper les journées présentant des comportements similaires en appliquant une méthode de **clustering non supervisé**. Ces profils journaliers caractéristiques permettront ensuite de mieux comprendre les habitudes de consommation des bâtiments et d'identifier différents comportements énergétiques.

---

## Approche suivie

La méthodologie est organisée en plusieurs étapes :

1. **Chargement des séries temporelles**
   - Lecture des fichiers de consommation énergétique des bâtiments individuels.
   - Sélection de la variable de consommation électrique étudiée.
   - Vérification de la fréquence temporelle des données.

2. **Transformation des séries temporelles en profils journaliers**
   - Les séries annuelles (pas de temps de 15 minutes) sont restructurées sous forme de matrices :
   
   \[
   \text{jours} \times \text{pas horaires}
   \]
   
   Chaque ligne représente alors un profil de consommation sur une journée.

3. **Prétraitement des profils**
   - Normalisation des profils journaliers afin de comparer les formes de consommation indépendamment du niveau énergétique absolu.
   - Cette étape permet de détecter des comportements similaires même lorsque les bâtiments ont des consommations différentes.

4. **Détermination du nombre optimal de clusters**
   - Plusieurs valeurs du nombre de groupes sont testées.
   - Le score de silhouette est utilisé pour mesurer la qualité de séparation des clusters.

5. **Application du clustering**
   - Utilisation de l'algorithme **K-Means** pour regrouper les jours présentant des profils similaires.
   - Chaque cluster représente un **jour type de consommation**.

6. **Analyse et visualisation des résultats**
   - Visualisation des centroïdes des clusters correspondant aux journées représentatives.
   - Analyse de la fréquence d'apparition de chaque type de journée.

---

## Données utilisées

Les données proviennent des séries temporelles individuelles des bâtiments du dataset **ResStock**.

Chaque bâtiment contient une série de consommation énergétique avec un pas de temps de 15 minutes :

- 96 mesures par jour ;
- 365 jours par année ;
- soit environ 35 040 points temporels par bâtiment.

La variable étudiée est :


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import matplotlib.pyplot as plt
import plotly.graph_objects as go
from kneed import KneeLocator

ROOT = Path().resolve().parent.parent

DATA_RAW       = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'
FIGURES        = ROOT / 'reports' / 'figures'


df = pd.read_parquet(DATA_RAW / "347201-0.parquet")
df.head()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

dff = (
 	pd.read_parquet(DATA_RAW / "347201-0.parquet")
   	.assign(timestamp=lambda x: x["timestamp"] - pd.Timedelta("15m"))
   	.set_index("timestamp")
   	.loc[:, lambda x:
x.columns.str.match(r"out\.electricity\..*\.energy_consumption\.\.kwh")]
   	.loc[:, lambda x: x.ne(0).any(axis=0)]
   	.rename(columns=lambda x: x[16:-24])
   	.drop(columns="net")
)

dfh = dff.resample("h").sum()
dfd = dfh.resample("D").sum()

fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)
dfd.drop(columns="total").plot(ax=ax, kind="area", cmap="tab20")
plt.legend(loc="upper right")
plt.show()
plt.close(fig)

piv = dfh.pivot_table(index=dfh.index.normalize(),
columns=dfh.index.hour, values="total")
cmap = (piv.index.day_of_week // 5).map({0: "tab:blue", 1: "tab:red"})
fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)
piv.T.plot(ax=ax, color=cmap, alpha=0.1, legend=False)
plt.show()


## Extraire les jours

In [ ]:
df.shape

In [ ]:
COL = "out.electricity.total.energy_consumption..kwh"

# Série de consommation
ts = df[COL].values

# Nombre de mesures par jour (15 min)
steps_per_day = 96

# Récupération des timestamps
if "timestamp" in df.columns:
    dates = pd.to_datetime(df["timestamp"])

elif isinstance(df.index, pd.DatetimeIndex):
    dates = pd.to_datetime(df.index)

else:
    dates = pd.date_range( start="2018-01-01",periods=len(df),freq="15min")

# Nombre de jours complets
n_days = len(ts) // steps_per_day

# Garder uniquement les jours complets
ts = ts[: n_days * steps_per_day]
dates = dates[: n_days * steps_per_day]

# Transformation en matrice :
# lignes = jours
# colonnes = pas de temps dans la journée
days = ts.reshape(n_days, steps_per_day)

# Une date par jour
day_dates = (
    pd.Series(dates)
    .groupby(pd.Series(np.arange(len(dates))) // steps_per_day)
    .first()
    .dt.normalize()
    .values
)

print("Nombre de jours :", n_days)
print("Shape matrice jours :", days.shape)
print("Première date :", day_dates[0])

In [ ]:
# Supprimer les jours avec NaN
mask_valid = ~np.isnan(days).any(axis=1)
print(f"Jours invalides supprimés : {(~mask_valid).sum()} / {len(days)}")

days_clean = days[mask_valid]
day_dates_clean = day_dates[mask_valid]

## Normalisation des profils journaliers

In [ ]:
# Option A : StandardScaler global (garde le niveau d'amplitude)
scaler_global = StandardScaler()
days_scaled_global = scaler_global.fit_transform(days_clean)

# Option B : Normalisation par jour (forme du profil, z-score intra-jour)
day_mean = days_clean.mean(axis=1, keepdims=True)
day_std  = days_clean.std(axis=1, keepdims=True)
day_std[day_std == 0] = 1e-8  # éviter division par 0 (jours plats)

days_scaled_shape = (days_clean - day_mean) / day_std

# --> on choisit l'option B pour la suite (jours types = formes de profils)
X = days_scaled_shape
Y = days_scaled_global 


## Choix du nombre de clusters (méthode du coude + silhouette)

In [ ]:
def plot_cluster_profiles(X, title1,title2):

    k_range = range(2, 15)
    inertias = []
    silhouettes = []

    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X)
        inertias.append(km.inertia_)
        sil = silhouette_score(X, labels, sample_size=5000, random_state=42)  # sample_size pour accélérer si beaucoup de jours
        silhouettes.append(sil)
        print(f"k={k:2d} | inertia={km.inertia_:10.1f} | silhouette={sil:.4f}")
    k_opt = KneeLocator(
    list(k_range),
    inertias,
    curve="convex",
    direction="decreasing"
    ).elbow

    print(f"K optimal selon la méthode du coude : {k_opt}")    

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(list(k_range), inertias, marker='o')
    axes[0].set_xlabel("k")
    axes[0].set_ylabel("Inertie")
    axes[0].set_title(title1)

    axes[1].plot(list(k_range), silhouettes, marker='o', color='orange')
    axes[1].set_xlabel("k")
    axes[1].set_ylabel("Silhouette score")
    axes[1].set_title(title2)

    plt.tight_layout()
    plt.savefig(FIGURES / "kmeans_elbow_silhouette.png", dpi=150)
    plt.show()
    return k_opt

k_forme=plot_cluster_profiles(Y, "Méthode du coude (inertie) - Clustering des jours types", "Silhouette score - Clustering des jours types")
k_amplet=plot_cluster_profiles(X, "Méthode du coude (inertie) - Clustering des jours types (niveau d'amplitude)", "Silhouette score - Clustering des jours types (niveau d'amplitude)")


## Clustering final avec le k choisi

In [ ]:
def cluster_days(k):

    kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
    cluster_labels = kmeans.fit_predict(X)

    results = pd.DataFrame({
        "date": day_dates_clean,
        "cluster": cluster_labels
    })

    results["weekday"] = results["date"].dt.day_name()
    results["is_weekend"] = results["date"].dt.dayofweek >= 5
    results["month"] = results["date"].dt.month

    print(results["cluster"].value_counts().sort_index())

    return kmeans, cluster_labels, results
kmeans_forme, cluster_labels_forme, results_forme =cluster_days(k_forme)
kmeans_amplet, cluster_labels_amplet, results_amplet=cluster_days(k_amplet)
print(results_forme)
print(results_amplet)

## Visualisation : profil moyen par cluster (courbes)

In [ ]:

def plot_interactive_cluster_profiles( cluster_labels, k_final,titre):

    fig = go.Figure()

    hours = np.linspace(0, 24, steps_per_day)

    # Traces
    for c in range(k_final):
        mask = cluster_labels == c

        cluster_profiles = days_clean[mask]
        mean_profile = cluster_profiles.mean(axis=0)
        std_profile = cluster_profiles.std(axis=0)

        # Courbe moyenne
        fig.add_trace(
            go.Scatter(
                x=hours,
                y=mean_profile,
                mode="lines",
                name=f"Cluster {c} (n={mask.sum()})",
                visible=(c == 0),
                line=dict(width=3)
            )
        )

        # Zone ±1σ
        fig.add_trace(
            go.Scatter(
                x=np.concatenate([hours, hours[::-1]]),
                y=np.concatenate([
                    mean_profile + std_profile,
                    (mean_profile - std_profile)[::-1]
                ]),
                fill="toself",
                mode="lines",
                opacity=0.15,
                name=f"Std Cluster {c}",
                visible=(c == 0),
                showlegend=False,
                hoverinfo="skip"
            )
        )

    # Boutons
    buttons = []

    for c in range(k_final):
        visibility = []

        for i in range(k_final):
            visibility += [i == c, i == c]

        buttons.append(
            dict(
                label=f"Cluster {c}",
                method="update",
                args=[
                    {"visible": visibility},
                    {"title": f"{titre} - Cluster {c}"}
                ]
            )
        )

    # Afficher tous les clusters
    buttons.append(
        dict(
            label="Tous les clusters",
            method="update",
            args=[
                {"visible": [True] * (2 * k_final)},
                {"title": f"{titre} (k={k_final})"}
            ]
        )
    )

    fig.update_layout(
        title=titre,
        xaxis_title="Heure de la journée",
        yaxis_title="Consommation électrique (kWh)",
        template="plotly_white",
        width=1000,
        height=600,
        updatemenus=[
            dict(
                buttons=buttons,
                direction="down",
                x=1.05,
                y=1,
                showactive=True
            )
        ]
    )

    fig.show()

    return 0

plot_interactive_cluster_profiles(
    cluster_labels=cluster_labels_amplet,
    k_final=k_amplet,
    titre="Profil journalier par rapport à la quantite d'energie consommée"
)
plot_interactive_cluster_profiles(
    cluster_labels=cluster_labels_forme,
    k_final=k_forme,
    titre="Profil journalier par rapport à la forme d'energie consommée"
)

In [ ]:
def plot_cluster_calendar_composition(results, titre):

    # Répartition semaine vs weekend
    comp_weekend = pd.crosstab(
        results["cluster"], 
        results["is_weekend"], 
        normalize="index"
    ) * 100

    comp_weekend.columns = ["Semaine (%)", "Weekend (%)"]

    print("Répartition semaine / weekend :")
    print(comp_weekend.round(1))
    print()

    # Répartition par mois
    comp_month = pd.crosstab(
        results["cluster"], 
        results["month"], 
        normalize="index"
    ) * 100

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Weekend
    comp_weekend.plot(
        kind="bar",
        stacked=True,
        ax=axes[0],
        colormap="Set2"
    )

    axes[0].set_title("Répartition semaine / weekend par cluster")
    axes[0].set_ylabel("%")
    axes[0].set_xlabel("Cluster")
    axes[0].legend(title="")

    # Mois
    sns.heatmap(
        comp_month,
        annot=True,
        fmt=".0f",
        cmap="YlOrRd",
        ax=axes[1]
    )

    axes[1].set_title("Répartition mensuelle par cluster (%)")
    axes[1].set_xlabel("Mois")
    axes[1].set_ylabel("Cluster")

    fig.suptitle(titre, fontsize=14)

    plt.tight_layout()
    plt.savefig(
        FIGURES / "cluster_calendar_composition.png",
        dpi=150,
        bbox_inches="tight"
    )
    plt.show()

    return comp_weekend, comp_month
plot_cluster_calendar_composition(results_amplet, "Composition temporelle des clusters(amplet)")
plot_cluster_calendar_composition(results_forme, "Composition temporelle des clusters(forme)")


In [ ]:
results_sorted = results.sort_values("date").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(16, 3))
scatter = ax.scatter(results_sorted["date"], [1]*len(results_sorted),
                      c=results_sorted["cluster"], cmap="tab10", s=15)
ax.set_yticks([])
ax.set_title("Assignation de cluster au fil de l'année")
plt.colorbar(scatter, ax=ax, label="Cluster", ticks=range(K_FINAL))
plt.tight_layout()
plt.savefig(FIGURES / "cluster_timeline.png", dpi=150)
plt.show()

In [ ]:
'''
results.to_parquet(DATA_PROCESSED / "clustering_jours_types.parquet", index=False)

# Sauvegarde du modèle et du scaler pour réutilisation
import joblib
joblib.dump(kmeans, DATA_PROCESSED / "kmeans_jours_types.joblib")

results.head()

'''